# PatrolIQ - Feature Engineering
## Notebook 3: Advanced Feature Creation for Machine Learning

**Objective:**
- Create crime severity scores
- Encode categorical features
- Normalize and scale numeric features
- Prepare final ML-ready dataset

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 1: Load Processed Dataset

In [2]:
# Load processed data
PROCESSED_DATA_PATH = '../data/processed/chicago_crimes_processed.csv'

print("Loading processed dataset...")
df = pd.read_csv(PROCESSED_DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Loading processed dataset...
Dataset shape: (498569, 34)
Columns: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location', 'Crime_Year', 'Month', 'Day', 'Hour', 'Day_of_Week', 'Day_Name', 'Month_Name', 'Is_Weekend', 'Is_Late_Night', 'Is_Rush_Hour', 'Season', 'Time_of_Day']


In [3]:
df.head()

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Day,Hour,Day_of_Week,Day_Name,Month_Name,Is_Weekend,Is_Late_Night,Is_Rush_Hour,Season,Time_of_Day
0,14111454,JK150303,2026-02-14,090XX S HOUSTON AVE,051A,ASSAULT,AGGRAVATED - HANDGUN,SIDEWALK,0,1,...,14,0,5,Saturday,February,1,1,0,Winter,Night
1,14111080,JK149857,2026-02-14,048XX N FRANCISCO AVE,0460,BATTERY,SIMPLE,RESIDENCE,1,0,...,14,0,5,Saturday,February,1,1,0,Winter,Night
2,14110206,JK148883,2026-02-14,063XX S RACINE AVE,5002,OTHER OFFENSE,OTHER VEHICLE OFFENSE,STREET,0,0,...,14,0,5,Saturday,February,1,1,0,Winter,Night
3,14111117,JK149856,2026-02-14,002XX W 23RD ST,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,0,0,...,14,0,5,Saturday,February,1,1,0,Winter,Night
4,14110668,JK149416,2026-02-14,047XX S KARLOV AVE,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,0,0,...,14,0,5,Saturday,February,1,1,0,Winter,Night


## Step 2: Create Crime Severity Score

In [4]:
# Define crime severity mapping based on crime seriousness
# Scale: 1 (Minor) to 5 (Very Severe)

crime_severity_map = {
    # Very Severe (5)
    'HOMICIDE': 5,
    'CRIM SEXUAL ASSAULT': 5,
    'KIDNAPPING': 5,
    'HUMAN TRAFFICKING': 5,
    
    # Severe (4)
    'ROBBERY': 4,
    'ASSAULT': 4,
    'ARSON': 4,
    'BURGLARY': 4,
    'WEAPONS VIOLATION': 4,
    'SEX OFFENSE': 4,
    
    # Moderate (3)
    'BATTERY': 3,
    'MOTOR VEHICLE THEFT': 3,
    'NARCOTICS': 3,
    'CRIMINAL DAMAGE': 3,
    'CRIMINAL TRESPASS': 3,
    'INTIMIDATION': 3,
    'STALKING': 3,
    
    # Minor (2)
    'THEFT': 2,
    'DECEPTIVE PRACTICE': 2,
    'INTERFERENCE WITH PUBLIC OFFICER': 2,
    'PUBLIC PEACE VIOLATION': 2,
    'OFFENSE INVOLVING CHILDREN': 2,
    'LIQUOR LAW VIOLATION': 2,
    
    # Very Minor (1)
    'GAMBLING': 1,
    'OBSCENITY': 1,
    'PUBLIC INDECENCY': 1,
    'PROSTITUTION': 1,
    'OTHER OFFENSE': 1,
    'NON-CRIMINAL': 1
}

# Apply severity mapping (default to 2 for unmapped crimes)
df['Crime_Severity'] = df['Primary Type'].map(crime_severity_map).fillna(2).astype(int)

print("Crime severity scores created!")
print(f"\nSeverity distribution:")
print(df['Crime_Severity'].value_counts().sort_index())

Crime severity scores created!

Severity distribution:
Crime_Severity
1     34673
2    158018
3    209365
4     95251
5      1262
Name: count, dtype: int64


In [5]:
# Show examples of crimes with severity scores
severity_examples = df.groupby('Primary Type')['Crime_Severity'].first().sort_values(ascending=False)
print("\n=== Crime Types with Severity Scores ===")
print(severity_examples.head(20))


=== Crime Types with Severity Scores ===
Primary Type
HOMICIDE                    5
KIDNAPPING                  5
HUMAN TRAFFICKING           5
ARSON                       4
SEX OFFENSE                 4
ROBBERY                     4
ASSAULT                     4
WEAPONS VIOLATION           4
BURGLARY                    4
CRIMINAL TRESPASS           3
STALKING                    3
INTIMIDATION                3
CRIMINAL DAMAGE             3
BATTERY                     3
MOTOR VEHICLE THEFT         3
NARCOTICS                   3
DECEPTIVE PRACTICE          2
OTHER NARCOTIC VIOLATION    2
THEFT                       2
PUBLIC PEACE VIOLATION      2
Name: Crime_Severity, dtype: int32


## Step 3: Additional Feature Engineering

In [6]:
# Create time-based binary features
print("Creating additional temporal features...")

# Time of day categories
def categorize_time_of_day(hour):
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 22:
        return 'Evening'
    else:
        return 'Night'

df['Time_of_Day'] = df['Hour'].apply(categorize_time_of_day)

# Late night flag (10 PM to 4 AM - high-risk hours)
df['Is_Late_Night'] = df['Hour'].apply(lambda x: 1 if x >= 22 or x < 4 else 0)

# Rush hour flag (7-9 AM and 5-7 PM)
df['Is_Rush_Hour'] = df['Hour'].apply(lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0)

print("✓ Time_of_Day created")
print("✓ Is_Late_Night flag created")
print("✓ Is_Rush_Hour flag created")

Creating additional temporal features...
✓ Time_of_Day created
✓ Is_Late_Night flag created
✓ Is_Rush_Hour flag created


In [7]:
# Create location density features
print("\nCreating geographic features...")

# District crime density
district_counts = df['District'].value_counts().to_dict()
df['District_Crime_Count'] = df['District'].map(district_counts)

# Ward crime density
ward_counts = df['Ward'].value_counts().to_dict()
df['Ward_Crime_Count'] = df['Ward'].map(ward_counts)

print("✓ District_Crime_Count created")
print("✓ Ward_Crime_Count created")


Creating geographic features...
✓ District_Crime_Count created
✓ Ward_Crime_Count created


In [8]:
# Display new features
new_features = ['Crime_Severity', 'Time_of_Day', 'Is_Late_Night', 'Is_Rush_Hour',
                'District_Crime_Count', 'Ward_Crime_Count']
df[new_features].head(10)

,Crime_Severity,Time_of_Day,Is_Late_Night,Is_Rush_Hour,District_Crime_Count,Ward_Crime_Count
0,4,Night,1,0,26927,10536
1,3,Night,1,0,10690,5839
2,1,Night,1,0,21009,15361
3,2,Night,1,0,22009,6258
4,3,Night,1,0,32658,7672
5,3,Night,1,0,26221,12125
6,3,Night,1,0,28615,14762
7,2,Night,1,0,26914,17860
8,1,Night,1,0,26927,14294
9,3,Night,1,0,15725,15160


## Step 4: Encode Categorical Features

In [9]:
# Label encode crime types
print("Encoding categorical features...\n")

le_crime = LabelEncoder()
df['Primary_Type_Encoded'] = le_crime.fit_transform(df['Primary Type'])

# Save mapping for later use
crime_type_mapping = dict(zip(le_crime.classes_, le_crime.transform(le_crime.classes_)))
print(f"✓ Primary_Type_Encoded created ({len(crime_type_mapping)} unique crime types)")

# Encode time of day
le_time = LabelEncoder()
df['Time_of_Day_Encoded'] = le_time.fit_transform(df['Time_of_Day'])
print("✓ Time_of_Day_Encoded created")

# Encode season
le_season = LabelEncoder()
df['Season_Encoded'] = le_season.fit_transform(df['Season'])
print("✓ Season_Encoded created")

# Encode day name
le_day = LabelEncoder()
df['Day_Name_Encoded'] = le_day.fit_transform(df['Day_Name'])
print("✓ Day_Name_Encoded created")

Encoding categorical features...

✓ Primary_Type_Encoded created (31 unique crime types)
✓ Time_of_Day_Encoded created
✓ Season_Encoded created
✓ Day_Name_Encoded created


In [10]:
# Show sample of encoded features
encoded_cols = ['Primary Type', 'Primary_Type_Encoded', 'Time_of_Day', 'Time_of_Day_Encoded']
df[encoded_cols].head(10)

,Primary Type,Primary_Type_Encoded,Time_of_Day,Time_of_Day_Encoded
0,ASSAULT,1,Night,3
1,BATTERY,2,Night,3
2,OTHER OFFENSE,22,Night,3
3,DECEPTIVE PRACTICE,8,Night,3
4,CRIMINAL DAMAGE,5,Night,3
5,CRIMINAL DAMAGE,5,Night,3
6,CRIMINAL DAMAGE,5,Night,3
7,THEFT,29,Night,3
8,OTHER OFFENSE,22,Night,3
9,MOTOR VEHICLE THEFT,16,Night,3


## Step 5: Normalize Geographic Features

In [11]:
# Scale latitude and longitude for clustering
print("Normalizing geographic coordinates...\n")

scaler_geo = StandardScaler()
df[['Latitude_Scaled', 'Longitude_Scaled']] = scaler_geo.fit_transform(
    df[['Latitude', 'Longitude']]
)

print("✓ Latitude_Scaled created")
print("✓ Longitude_Scaled created")

print(f"\nOriginal coordinate ranges:")
print(f"Latitude: {df['Latitude'].min():.4f} to {df['Latitude'].max():.4f}")
print(f"Longitude: {df['Longitude'].min():.4f} to {df['Longitude'].max():.4f}")

print(f"\nScaled coordinate ranges:")
print(f"Latitude_Scaled: {df['Latitude_Scaled'].min():.4f} to {df['Latitude_Scaled'].max():.4f}")
print(f"Longitude_Scaled: {df['Longitude_Scaled'].min():.4f} to {df['Longitude_Scaled'].max():.4f}")

Normalizing geographic coordinates...

✓ Latitude_Scaled created
✓ Longitude_Scaled created

Original coordinate ranges:
Latitude: 41.6446 to 42.0226
Longitude: -87.9346 to -87.5245

Scaled coordinate ranges:
Latitude_Scaled: -2.3354 to 2.0333
Longitude_Scaled: -4.5145 to 2.4344


## Step 6: Create ML Feature Matrix

In [12]:
# Select features for machine learning
print("Creating ML feature matrix...\n")

# Geographic features
geo_features = ['Latitude', 'Longitude', 'Latitude_Scaled', 'Longitude_Scaled',
                'District', 'Ward', 'Community Area', 
                'District_Crime_Count', 'Ward_Crime_Count']

# Temporal features
temporal_features = ['Hour', 'Day_of_Week', 'Month', 'Is_Weekend', 
                     'Is_Late_Night', 'Is_Rush_Hour',
                     'Time_of_Day_Encoded', 'Season_Encoded', 'Day_Name_Encoded']

# Crime features
crime_features = ['Primary_Type_Encoded', 'Crime_Severity', 'Arrest']

# Combine all ML features
ml_features = geo_features + temporal_features + crime_features

# Create feature matrix
X = df[ml_features].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Total features: {len(ml_features)}")
print(f"\nFeature categories:")
print(f"  - Geographic: {len(geo_features)}")
print(f"  - Temporal: {len(temporal_features)}")
print(f"  - Crime: {len(crime_features)}")

Creating ML feature matrix...

Feature matrix shape: (498569, 21)
Total features: 21

Feature categories:
  - Geographic: 9
  - Temporal: 9
  - Crime: 3


In [13]:
# Display feature matrix
X.head()

,Latitude,Longitude,Latitude_Scaled,Longitude_Scaled,District,Ward,Community Area,District_Crime_Count,Ward_Crime_Count,Hour,...,Month,Is_Weekend,Is_Late_Night,Is_Rush_Hour,Time_of_Day_Encoded,Season_Encoded,Day_Name_Encoded,Primary_Type_Encoded,Crime_Severity,Arrest
0,41.731236,-87.549895,-1.333887,2.004499,4,10,46,26927,10536,0,...,2,1,1,0,3,3,2,1,4,0
1,41.969389,-87.700489,1.418769,-0.547594,20,40,4,10690,5839,0,...,2,1,1,0,3,3,2,2,3,1
2,41.779153,-87.654491,-0.780046,0.231917,7,16,67,21009,15361,0,...,2,1,1,0,3,3,2,22,1,0
3,41.850955,-87.633579,0.049866,0.586326,9,11,34,22009,6258,0,...,2,1,1,0,3,3,2,8,2,0
4,41.807233,-87.726013,-0.455496,-0.980152,8,14,57,32658,7672,0,...,2,1,1,0,3,3,2,5,3,0


In [14]:
# Check for missing values in feature matrix
print("Checking for missing values in feature matrix...")
missing_features = X.isnull().sum()
if missing_features.sum() == 0:
    print("✓ No missing values found!")
else:
    print("⚠ Missing values detected:")
    print(missing_features[missing_features > 0])

Checking for missing values in feature matrix...
✓ No missing values found!


## Step 7: Feature Statistics and Summary

In [15]:
# Statistical summary of all features
X.describe().T

,count,mean,std,min,25%,50%,75%,max
Latitude,498569.0,4.184664e+01,0.086518,41.644590,41.772296,41.864823,41.909483,42.022559
Longitude,498569.0,-8.766818e+01,0.059008,-87.934567,-87.709271,-87.660848,-87.626861,-87.524529
Latitude_Scaled,498569.0,-4.119631e-14,1.000001,-2.335381,-0.859302,0.210152,0.726346,2.033321
Longitude_Scaled,498569.0,4.595891e-13,1.000001,-4.514492,-0.696432,0.124201,0.700164,2.434368
District,498569.0,1.126818e+01,7.080784,1.000000,5.000000,10.000000,17.000000,31.000000
Ward,498569.0,2.311132e+01,13.938042,0.000000,10.000000,23.000000,34.000000,50.000000
Community Area,498569.0,3.628046e+01,21.559980,0.000000,22.000000,32.000000,53.000000,77.000000
District_Crime_Count,498569.0,2.414675e+04,5480.395548,22.000000,19576.000000,26221.000000,26927.000000,32658.000000
Ward_Crime_Count,498569.0,1.242247e+04,5429.953798,1.000000,7764.000000,13455.000000,16349.000000,23560.000000
Hour,498569.0,1.249824e+01,6.828375,0.000000,8.000000,13.000000,18.000000,23.000000


In [16]:
# Feature data types
print("\n=== Feature Data Types ===")
print(X.dtypes)


=== Feature Data Types ===
Latitude                float64
Longitude               float64
Latitude_Scaled         float64
Longitude_Scaled        float64
District                  int64
Ward                      int64
Community Area            int64
District_Crime_Count      int64
Ward_Crime_Count          int64
Hour                      int64
Day_of_Week               int64
Month                     int64
Is_Weekend                int64
Is_Late_Night             int64
Is_Rush_Hour              int64
Time_of_Day_Encoded       int32
Season_Encoded            int32
Day_Name_Encoded          int32
Primary_Type_Encoded      int32
Crime_Severity            int32
Arrest                    int64
dtype: object


## Step 8: Save Enhanced Dataset

In [17]:
# Save the enhanced dataset with all features
import os

# Update processed CSV with new features
print("Saving enhanced dataset...")
df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"\n✅ Enhanced dataset saved to {PROCESSED_DATA_PATH}")
print(f"Total columns: {len(df.columns)}")
print(f"Total rows: {len(df):,}")
print(f"File size: {os.path.getsize(PROCESSED_DATA_PATH) / (1024*1024):.2f} MB")

Saving enhanced dataset...

✅ Enhanced dataset saved to ../data/processed/chicago_crimes_processed.csv
Total columns: 43
Total rows: 498,569
File size: 163.72 MB


## Step 9: Feature Engineering Summary

In [18]:
print("="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)

print("\n1. NEW FEATURES CREATED:")
print("   ✓ Crime_Severity (1-5 scale based on crime type)")
print("   ✓ Time_of_Day (Morning/Afternoon/Evening/Night)")
print("   ✓ Is_Late_Night (10 PM - 4 AM flag)")
print("   ✓ Is_Rush_Hour (7-9 AM, 5-7 PM flag)")
print("   ✓ District_Crime_Count (crime density per district)")
print("   ✓ Ward_Crime_Count (crime density per ward)")

print("\n2. ENCODED FEATURES:")
print(f"   ✓ Primary_Type_Encoded ({len(crime_type_mapping)} crime types)")
print("   ✓ Time_of_Day_Encoded (4 categories)")
print("   ✓ Season_Encoded (4 seasons)")
print("   ✓ Day_Name_Encoded (7 days)")

print("\n3. SCALED FEATURES:")
print("   ✓ Latitude_Scaled (standardized)")
print("   ✓ Longitude_Scaled (standardized)")

print(f"\n4. FINAL ML FEATURE SET:")
print(f"   - Total features: {len(ml_features)}")
print(f"   - Geographic features: {len(geo_features)}")
print(f"   - Temporal features: {len(temporal_features)}")
print(f"   - Crime features: {len(crime_features)}")

print("\n5. DATA QUALITY:")
print(f"   ✓ No missing values in feature matrix")
print(f"   ✓ All features properly scaled/encoded")
print(f"   ✓ Ready for unsupervised learning")

print("\n" + "="*80)
print("✅ FEATURE ENGINEERING COMPLETE!")
print("="*80)

FEATURE ENGINEERING SUMMARY

1. NEW FEATURES CREATED:
   ✓ Crime_Severity (1-5 scale based on crime type)
   ✓ Time_of_Day (Morning/Afternoon/Evening/Night)
   ✓ Is_Late_Night (10 PM - 4 AM flag)
   ✓ Is_Rush_Hour (7-9 AM, 5-7 PM flag)
   ✓ District_Crime_Count (crime density per district)
   ✓ Ward_Crime_Count (crime density per ward)

2. ENCODED FEATURES:
   ✓ Primary_Type_Encoded (31 crime types)
   ✓ Time_of_Day_Encoded (4 categories)
   ✓ Season_Encoded (4 seasons)
   ✓ Day_Name_Encoded (7 days)

3. SCALED FEATURES:
   ✓ Latitude_Scaled (standardized)
   ✓ Longitude_Scaled (standardized)

4. FINAL ML FEATURE SET:
   - Total features: 21
   - Geographic features: 9
   - Temporal features: 9
   - Crime features: 3

5. DATA QUALITY:
   ✓ No missing values in feature matrix
   ✓ All features properly scaled/encoded
   ✓ Ready for unsupervised learning

✅ FEATURE ENGINEERING COMPLETE!


## ✅ Feature Engineering Complete!

**Achievements:**
- Created crime severity scoring system
- Generated temporal and geographic features
- Encoded all categorical variables
- Normalized coordinates for distance-based algorithms
- Built ML-ready feature matrix with 21 features

**Next Step:** Proceed to `04_geo_clustering.ipynb` for geographic hotspot analysis